# ETL Pipeline: Livestock Intelligence
## Fase 1 — EXTRACT
**Laporan Praktikum Teknologi Perekayasaan Data · Kelompok 1 (3SI1)**

| Anggota | NIM |
|---|---|
| Anggita Cristin Meylani | 222312982 |
| Clarisse De Delgada M. Soares | 222313033 |
| M Rezky Raya Kilwouw | 222313190 |
| Nyimas Virna S. L. R | 222313307 |

---

### Scope Fase Extract
Fase ini **hanya** bertugas menarik data mentah (*raw*) dari ketiga sumber dan memuatnya ke database staging.  
**Tidak ada pembersihan, imputasi, maupun transformasi** di fase ini.

| # | Sumber Data | Metode | Target Database |
|---|---|---|---|
| 1 | **BPS** | API Scraping (4 variabel) + Dummy Generator (11 variabel) | `bps_db` (PostgreSQL) → `staging_db` |
| 2 | **iSIKHNAS** | Read dari MySQL (`isikhnas_db`) | `staging_db` (PostgreSQL) |
| 3 | **PIHPS** | Read dari file Excel (`final_data.xlsx`) | `staging_db` (PostgreSQL) |

> **Catatan Airflow:** Pipeline ini dirancang agar setiap section dapat dijadikan DAG task terpisah di Apache Airflow. Struktur fungsi sudah disiapkan agar mudah di-wrap sebagai `PythonOperator`.


In [ ]:
# ============================================================
# CELL 1 · IMPORT LIBRARY & KONFIGURASI KONEKSI
# ============================================================

import pandas as pd
import numpy as np
import random
import requests
import time
import warnings
from tqdm import tqdm
from sqlalchemy import create_engine, text, inspect

warnings.filterwarnings('ignore')
random.seed(42)   # Reproducibility untuk dummy generator

# -----------------------------------------------------------
# Konfigurasi PostgreSQL
# Sesuaikan PG_PASS dengan password PostgreSQL lokal Anda
# -----------------------------------------------------------
PG_HOST = 'localhost'
PG_PORT = '5432'
PG_USER = 'postgres'
PG_PASS = 'postgres'      # ← GANTI jika password berbeda

# -----------------------------------------------------------
# Konfigurasi MySQL (iSIKHNAS via XAMPP/phpMyAdmin)
# -----------------------------------------------------------
MYSQL_HOST = 'localhost'
MYSQL_PORT = '3306'
MYSQL_USER = 'root'
MYSQL_PASS = ''            # ← Kosong = default XAMPP; ganti jika ada password

# -----------------------------------------------------------
# Konfigurasi Path File PIHPS
# -----------------------------------------------------------
PIHPS_FILE_PATH = '../DATA/PIHPS/final_data.xlsx'   # ← Sesuaikan path

# -----------------------------------------------------------
# Inisialisasi Engine
# -----------------------------------------------------------
try:
    engine_bps     = create_engine(f'postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/bps_db')
    engine_staging = create_engine(f'postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/staging_db')
    engine_mysql   = create_engine(f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/isikhnas_db')
    print("✅ Semua engine database berhasil diinisialisasi.")
    print(f"   • bps_db     → postgresql://{PG_HOST}:{PG_PORT}/bps_db")
    print(f"   • staging_db → postgresql://{PG_HOST}:{PG_PORT}/staging_db")
    print(f"   • isikhnas_db → mysql://{MYSQL_HOST}:{MYSQL_PORT}/isikhnas_db")
except Exception as e:
    print(f"❌ Gagal menginisialisasi engine: {e}")


---
## 1. Extract Data BPS

### 1.1 Peta Variabel

| Variabel | Satuan | Metode |
|---|---|---|
| `provinsi` | — | Scraping API |
| `tahun` | — | Scraping API |
| `jumlah_penduduk` | jiwa | Scraping API (SIMDASI) |
| `populasi_sapi` | ekor | Scraping API (SIMDASI) |
| `populasi_ayam` | ekor | Scraping API (SIMDASI) |
| `produksi_daging_ayam` | kg | Scraping API (SIMDASI) |
| `produksi_daging_sapi` | ton | **Dummy** (data BPS tidak tersedia lengkap) |
| `konsumsi_daging_sapi` | kg/kapita | **Dummy** |
| `konsumsi_daging_ayam` | kg/kapita | **Dummy** |
| `permintaan_daging_sapi` | ekor | **Dummy** |
| `permintaan_daging_ayam` | ekor | **Dummy** |
| `jumlah_ternak_sapi_potong` | ekor | **Dummy** |
| `jumlah_ternak_ayam_potong` | ekor | **Dummy** |
| `harga_baseline_sapi` | Rp/kg | **Dummy** |
| `harga_baseline_ayam` | Rp/kg | **Dummy** |
| `growth_populasi` | desimal | **Dummy** |
| `status_supply` | kategori | **Dummy** |

### 1.2 Strategi Scraping
- Sumber: **BPS Web API** (`webapi.bps.go.id`) endpoint SIMDASI
- Pendekatan: Ambil data **nasional** sekali per tahun per komoditas → parse semua provinsi dari 1 response (lebih efisien daripada hit API per provinsi)
- Acuan wilayah: **nama provinsi** (bukan kode wilayah)

### 1.3 Skema Target di `bps_db` (PostgreSQL)
```sql
ref_wilayah           -- Master provinsi (PK: kode_wilayah surrogate 1–38)
ref_komoditas         -- Master komoditas Sapi & Ayam (PK: id_komoditas)
tr_demografi_tahunan  -- Populasi penduduk & growth per provinsi per tahun
tr_statistik_peternakan -- Semua metrik peternakan per provinsi-tahun-komoditas
```


In [ ]:
# ============================================================
# CELL 3 · KONFIGURASI API BPS & FUNGSI INTI SCRAPING
# ============================================================

API_KEY = "d328d5f200379241367024848106698e"
YEARS   = [2020, 2021, 2022, 2023, 2024, 2025]

# -----------------------------------------------------------
# Definisi 4 komoditas yang di-scrape
# (produksi_daging_sapi TIDAK di-scrape → masuk dummy)
# -----------------------------------------------------------
COMMODITIES_TO_SCRAPE = [
    {
        "name"      : "Jumlah Penduduk",
        "table_ids" : ["WVRlTTcySlZDa3lUcFp6czNwbHl4QT09"],
        "keywords"  : ["jumlah", "penduduk"],
        "col"       : "jumlah_penduduk",
    },
    {
        "name"      : "Populasi Sapi Potong",
        "table_ids" : ["S2ViU1dwVTlpSXRwU1MvendHN05Cdz09"],
        "keywords"  : ["populasi", "sapi", "potong"],
        "col"       : "populasi_sapi",
    },
    {
        "name"      : "Populasi Ayam Pedaging",
        "table_ids" : ["ckJyVXRMT05MWTNpaW9mdnVseFk0Zz09"],
        "keywords"  : ["populasi", "ayam", "pedaging"],
        "col"       : "populasi_ayam",
    },
    {
        "name"      : "Produksi Daging Ayam Pedaging",
        "table_ids" : ["dWhmNFl6WXYyR093R2NjTGM3NG9idz09"],
        "keywords"  : ["produksi", "daging", "ayam", "pedaging"],
        "col"       : "produksi_daging_ayam",
    },
]

# -----------------------------------------------------------
# Fungsi-fungsi inti scraping
# -----------------------------------------------------------

def _get(year: int, table_id: str, kode_wilayah: str = "0000000") -> dict | None:
    """Hit satu endpoint SIMDASI BPS. Return dict JSON atau None jika gagal."""
    url = (
        f"https://webapi.bps.go.id/v1/api/interoperabilitas/datasource/simdasi/"
        f"id/25/tahun/{year}/id_tabel/{table_id}/wilayah/{kode_wilayah}/key/{API_KEY}"
    )
    try:
        r = requests.get(url, timeout=30)
        return r.json()
    except Exception:
        return None


def _find_var_id(main_data: dict, keywords: list[str]) -> str | None:
    """
    Cari ID variabel target dalam metadata kolom BPS
    berdasarkan semua kata kunci (case-insensitive).
    """
    for var_id, info in main_data.get("kolom", {}).items():
        nama = str(info.get("nama_variabel", "")).lower()
        if all(kw in nama for kw in keywords):
            return var_id
    return None


def scrape_commodity_all_provinces(commodity: dict, year: int) -> pd.DataFrame:
    """
    Ambil data satu komoditas untuk satu tahun.
    Strategi: hit API nasional → temukan tabel+variabel yang valid →
    parse semua data provinsi dari response yang sama.
    Return DataFrame dengan kolom: provinsi, tahun, <col>.
    """
    rows = []

    for tbl_id in commodity["table_ids"]:
        raw = _get(year, tbl_id)
        if not raw or raw.get("status") != "OK":
            continue

        main_data = raw.get("data", [{}, {}])[1]
        var_id = _find_var_id(main_data, commodity["keywords"])
        if not var_id:
            continue

        # Berhasil menemukan tabel & variabel yang valid
        for item in main_data.get("data", []):
            nama_prov = item.get("label", "").strip()
            kode      = item.get("kode_wilayah", "")

            # Lewati baris "Indonesia" (total nasional)
            if not kode or nama_prov.lower() in ("indonesia", ""):
                continue

            val = None
            vars_data = item.get("variables", {})
            if var_id in vars_data:
                val = vars_data[var_id].get("value_raw", None)

            rows.append({
                "provinsi": nama_prov,
                "tahun"   : year,
                commodity["col"]: val,
            })

        # Tabel valid ditemukan, tidak perlu cek tabel alternatif
        break

    return pd.DataFrame(rows)


print("✅ Fungsi scraping berhasil didefinisikan.")
print(f"   Komoditas yang akan di-scrape: {[c['name'] for c in COMMODITIES_TO_SCRAPE]}")


---
### Pre-Flight Check: Verifikasi Ketersediaan API & Preview Data

> **Tujuan cell ini:** Memastikan API key valid, endpoint dapat diakses, dan data tersedia  
> untuk setiap komoditas di setiap tahun sebelum scraping penuh dijalankan.  
> **Cell ini dapat dihapus setelah verifikasi selesai.**


In [ ]:
# ============================================================
# CELL 5 · PRE-FLIGHT CHECK
# Hapus cell ini setelah verifikasi selesai
# ============================================================

print("=" * 65)
print(" PRE-FLIGHT CHECK — Ketersediaan API BPS")
print("=" * 65)

status_rows = []

for comm in COMMODITIES_TO_SCRAPE:
    row = {"Komoditas": comm["name"]}

    for y in YEARS:
        status = "❌"
        preview = None

        for tbl_id in comm["table_ids"]:
            raw = _get(y, tbl_id)
            if not raw or raw.get("status") != "OK":
                continue
            main_data = raw.get("data", [{}, {}])[1]
            var_id = _find_var_id(main_data, comm["keywords"])
            if not var_id:
                continue

            # Ambil 1 contoh nilai dari provinsi pertama yang ada nilainya
            for item in main_data.get("data", []):
                kode = item.get("kode_wilayah", "")
                if not kode or item.get("label", "").lower() in ("indonesia", ""):
                    continue
                val = item.get("variables", {}).get(var_id, {}).get("value_raw")
                if val is not None:
                    preview = f"✅ ({val})"
                    break
            if preview:
                status = preview
            break

        row[str(y)] = status if preview else "❌"
        time.sleep(0.3)   # Jaga rate-limit API

    status_rows.append(row)

df_check = pd.DataFrame(status_rows).set_index("Komoditas")
print()
print(df_check.to_string())
print()
print("Keterangan:")
print("  ✅ (nilai) = Endpoint aktif, data tersedia, contoh nilai ditampilkan")
print("  ❌         = Endpoint tidak responsif atau data tidak ditemukan")
print()
print("Jika semua ✅, lanjut ke Cell 7 (Scraping penuh).")
print("Jika ada ❌, cek API_KEY atau tabel ID di CELL 3.")


---
### 1.1 Scraping API BPS — Full Run

Proses ini meng-hit API BPS untuk **4 komoditas × 6 tahun**.  
Strategi efisien: 1 request per komoditas per tahun (bukan per provinsi),  
lalu seluruh data provinsi di-parse dari 1 response.

> **Estimasi waktu:** ~2–5 menit (tergantung kecepatan respons server BPS)


In [ ]:
# ============================================================
# CELL 7 · SCRAPING API BPS — FULL RUN
# ============================================================

print("🚀 Memulai scraping BPS...")
print(f"   Komoditas : {len(COMMODITIES_TO_SCRAPE)} variabel")
print(f"   Tahun     : {YEARS}")
print()

# Kumpulkan semua DataFrame per komoditas
commodity_dfs = {}

for comm in COMMODITIES_TO_SCRAPE:
    print(f"--- [{comm['name']}] ---")
    yearly_dfs = []

    for year in tqdm(YEARS, desc=f"  {comm['col']}", ncols=70):
        df_year = scrape_commodity_all_provinces(comm, year)
        if not df_year.empty:
            yearly_dfs.append(df_year)
        time.sleep(0.5)   # Jaga rate-limit

    if yearly_dfs:
        commodity_dfs[comm["col"]] = pd.concat(yearly_dfs, ignore_index=True)
        print(f"   ✅ {len(commodity_dfs[comm['col']])} baris terkumpul")
    else:
        print(f"   ❌ Tidak ada data berhasil diambil untuk {comm['name']}")

print()

# -----------------------------------------------------------
# Merge semua komoditas menjadi 1 DataFrame flat
# Key merge: provinsi + tahun
# -----------------------------------------------------------
if commodity_dfs:
    keys = list(commodity_dfs.keys())
    df_scraped = commodity_dfs[keys[0]]

    for col in keys[1:]:
        df_scraped = df_scraped.merge(
            commodity_dfs[col],
            on=["provinsi", "tahun"],
            how="outer"
        )

    # Normalisasi nama provinsi (strip spasi, title case konsisten)
    df_scraped["provinsi"] = df_scraped["provinsi"].str.strip()

    print(f"✅ Hasil scraping: {df_scraped.shape[0]} baris × {df_scraped.shape[1]} kolom")
    print(f"   Kolom: {list(df_scraped.columns)}")
    print()
    display(df_scraped.head(10))
else:
    print("⚠️  Tidak ada data scraping. Cek koneksi internet & API key.")
    df_scraped = pd.DataFrame(columns=["provinsi", "tahun",
                                        "jumlah_penduduk", "populasi_sapi",
                                        "populasi_ayam", "produksi_daging_ayam"])


---
### 1.2 Generate Data Dummy & Merge dengan Hasil Scraping

**Strategi merge:**
1. Generate dummy untuk **semua** 38 provinsi × 6 tahun (228 baris)
2. Merge dengan hasil scraping berdasarkan `provinsi` + `tahun`
3. Nilai dari scraping **menimpa (overwrite)** nilai dummy pada kolom yang bersesuaian

> Kolom yang diisi dummy: `produksi_daging_sapi`, `konsumsi_daging_sapi`, `konsumsi_daging_ayam`,  
> `permintaan_daging_sapi`, `permintaan_daging_ayam`, `jumlah_ternak_sapi_potong`,  
> `jumlah_ternak_ayam_potong`, `harga_baseline_sapi`, `harga_baseline_ayam`,  
> `growth_populasi`, `status_supply`


In [ ]:
# ============================================================
# CELL 9 · GENERATE DATA DUMMY & MERGE
# ============================================================

# Daftar 38 provinsi (sesuai standar nama BPS resmi)
PROVINSI_LIST = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau",
    "Kepulauan Riau", "Jambi", "Sumatera Selatan",
    "Kepulauan Bangka Belitung", "Bengkulu", "Lampung",
    "DKI Jakarta", "Jawa Barat", "Jawa Tengah", "DI Yogyakarta",
    "Jawa Timur", "Banten", "Bali",
    "Nusa Tenggara Barat", "Nusa Tenggara Timur",
    "Kalimantan Barat", "Kalimantan Tengah", "Kalimantan Selatan",
    "Kalimantan Timur", "Kalimantan Utara",
    "Sulawesi Utara", "Sulawesi Tengah", "Sulawesi Selatan",
    "Sulawesi Tenggara", "Gorontalo", "Sulawesi Barat",
    "Maluku", "Maluku Utara",
    "Papua", "Papua Barat", "Papua Selatan",
    "Papua Tengah", "Papua Pegunungan", "Papua Barat Daya",
]


def generate_bps_dummy(provinsi_list: list, years: list, seed: int = 42) -> pd.DataFrame:
    """
    Generate data dummy realistis untuk variabel BPS yang tidak tersedia di API.
    Missing value diinjeksi secara acak (±5%) untuk mensimulasikan kondisi nyata.
    """
    rng = random.Random(seed)
    data = []

    for prov in provinsi_list:
        # Base populasi manusia berbeda per provinsi (persisten antar tahun)
        base_pop = rng.randint(500_000, 50_000_000)

        for tahun in years:
            growth = round(rng.uniform(-0.02, 0.04), 4)
            jumlah_penduduk = int(base_pop * (1 + growth))

            # Populasi ternak (dipakai sebagai fallback jika scraping kosong)
            populasi_sapi = rng.randint(10_000, 500_000)
            populasi_ayam = rng.randint(50_000, 5_000_000)

            # Pemotongan ternak
            potong_sapi = int(populasi_sapi * rng.uniform(0.40, 0.70))
            potong_ayam = int(populasi_ayam * rng.uniform(0.50, 0.80))

            # Produksi daging (ton untuk sapi, kg untuk ayam)
            produksi_sapi = round(potong_sapi * rng.uniform(0.20, 0.30), 2)
            produksi_ayam = round(potong_ayam * rng.uniform(0.10, 0.20) * 1000, 2)  # dalam kg

            # Konsumsi per kapita (kg/tahun)
            konsumsi_sapi = round(rng.uniform(1.5, 3.5), 2)
            konsumsi_ayam = round(rng.uniform(8.0, 15.0), 2)

            # Permintaan (ekor) ≈ total kebutuhan / berat per ekor
            permintaan_sapi = int(jumlah_penduduk * konsumsi_sapi / 250)   # ~250 kg/ekor sapi
            permintaan_ayam = int(jumlah_penduduk * konsumsi_ayam / 1.5)   # ~1.5 kg/ekor ayam

            # Harga baseline pemerintah (Rp/kg)
            harga_sapi = rng.randint(90_000, 130_000)
            harga_ayam = rng.randint(20_000, 40_000)

            # Status supply berdasarkan rasio produksi vs permintaan
            ratio_sapi = produksi_sapi * 1000 / (permintaan_sapi * 250) if permintaan_sapi > 0 else 0
            if ratio_sapi >= 0.9:
                status = "Surplus"
            elif ratio_sapi >= 0.7:
                status = "Aman"
            elif ratio_sapi >= 0.5:
                status = "Waspada"
            else:
                status = "Defisit"

            row = {
                "provinsi"                  : prov,
                "tahun"                     : tahun,
                "jumlah_penduduk"           : jumlah_penduduk,
                "populasi_sapi"             : populasi_sapi,
                "populasi_ayam"             : populasi_ayam,
                "produksi_daging_sapi"      : produksi_sapi,
                "produksi_daging_ayam"      : produksi_ayam,
                "konsumsi_daging_sapi"      : konsumsi_sapi,
                "konsumsi_daging_ayam"      : konsumsi_ayam,
                "permintaan_daging_sapi"    : permintaan_sapi,
                "permintaan_daging_ayam"    : permintaan_ayam,
                "jumlah_ternak_sapi_potong" : potong_sapi,
                "jumlah_ternak_ayam_potong" : potong_ayam,
                "harga_baseline_sapi"       : harga_sapi,
                "harga_baseline_ayam"       : harga_ayam,
                "growth_populasi"           : growth,
                "status_supply"             : status,
            }

            # Inject missing value (±5%) — mensimulasikan kondisi data nyata
            for col_mv, prob in [
                ("produksi_daging_sapi", 0.05),
                ("permintaan_daging_ayam", 0.05),
                ("harga_baseline_sapi", 0.05),
                ("jumlah_ternak_sapi_potong", 0.03),
            ]:
                if rng.random() < prob:
                    row[col_mv] = None

            data.append(row)

    return pd.DataFrame(data)


# ------- Generate dummy -------
df_dummy = generate_bps_dummy(PROVINSI_LIST, YEARS)
print(f"✅ Dummy berhasil di-generate: {df_dummy.shape[0]} baris × {df_dummy.shape[1]} kolom")

# ------- Merge: dummy sebagai base, scraping sebagai overwrite -------
SCRAPE_COLS = ["jumlah_penduduk", "populasi_sapi", "populasi_ayam", "produksi_daging_ayam"]

if not df_scraped.empty:
    df_bps_full = df_dummy.merge(
        df_scraped[["provinsi", "tahun"] + SCRAPE_COLS],
        on=["provinsi", "tahun"],
        how="left",
        suffixes=("_dummy", "_api")
    )

    # Overwrite: gunakan nilai API jika ada, fallback ke dummy
    for col in SCRAPE_COLS:
        col_api   = col + "_api"
        col_dummy = col + "_dummy"
        if col_api in df_bps_full.columns:
            df_bps_full[col] = df_bps_full[col_api].combine_first(df_bps_full[col_dummy])
            df_bps_full.drop(columns=[col_api, col_dummy], inplace=True)

    print(f"✅ Merge selesai. Nilai dari scraping API menimpa dummy pada: {SCRAPE_COLS}")
else:
    # Jika scraping gagal total, gunakan dummy saja
    df_bps_full = df_dummy.copy()
    print("⚠️  Scraping gagal. Menggunakan data dummy penuh.")

print(f"   Shape akhir: {df_bps_full.shape}")
print()
display(df_bps_full.head(10))


---
### 1.3 Normalisasi ke 4 Tabel Relasional

Data flat BPS di-UNPIVOT menjadi skema relasional:

```
ref_wilayah            → Master provinsi (PK: kode_wilayah integer 1–38)
ref_komoditas          → Master komoditas (PK: id_komoditas; 1=Sapi, 2=Ayam)
tr_demografi_tahunan   → Penduduk & growth (PK: kode_wilayah, tahun)
tr_statistik_peternakan → Semua metrik ternak (PK: kode_wilayah, tahun, id_komoditas)
```

> `kode_wilayah` adalah **surrogate key** (integer 1–38 berurutan), bukan kode BPS.  
> Acuan join antar tabel menggunakan `kode_wilayah` ini.


In [ ]:
# ============================================================
# CELL 11 · NORMALISASI KE 4 TABEL RELASIONAL
# ============================================================

# ------- 1. ref_wilayah -------
provinsi_sorted = sorted(df_bps_full["provinsi"].unique())
df_ref_wilayah = pd.DataFrame({
    "kode_wilayah" : range(1, len(provinsi_sorted) + 1),
    "nama_provinsi": provinsi_sorted,
})
# Buat mapping provinsi → kode_wilayah untuk join berikutnya
prov_to_kode = dict(zip(df_ref_wilayah["nama_provinsi"], df_ref_wilayah["kode_wilayah"]))

print(f"✅ ref_wilayah     : {len(df_ref_wilayah)} baris")

# ------- 2. ref_komoditas -------
df_ref_komoditas = pd.DataFrame({
    "id_komoditas"  : [1, 2],
    "nama_komoditas": ["Sapi", "Ayam"],
    "satuan_berat"  : ["Ton", "Kg"],
})

print(f"✅ ref_komoditas   : {len(df_ref_komoditas)} baris")

# ------- 3. tr_demografi_tahunan -------
df_demografi = (
    df_bps_full[["provinsi", "tahun", "jumlah_penduduk", "growth_populasi"]]
    .copy()
)
df_demografi["kode_wilayah"] = df_demografi["provinsi"].map(prov_to_kode)
df_demografi = df_demografi[["kode_wilayah", "tahun", "jumlah_penduduk", "growth_populasi"]]

print(f"✅ tr_demografi    : {len(df_demografi)} baris")

# ------- 4. tr_statistik_peternakan (UNPIVOT Sapi & Ayam) -------
# Kolom yang berkaitan dengan SAPI (id_komoditas = 1)
cols_sapi = {
    "populasi_sapi"             : "populasi",
    "produksi_daging_sapi"      : "produksi",
    "konsumsi_daging_sapi"      : "konsumsi_per_kapita",
    "permintaan_daging_sapi"    : "permintaan_ekor",
    "jumlah_ternak_sapi_potong" : "jumlah_dipotong",
    "harga_baseline_sapi"       : "harga_baseline",
}

# Kolom yang berkaitan dengan AYAM (id_komoditas = 2)
cols_ayam = {
    "populasi_ayam"             : "populasi",
    "produksi_daging_ayam"      : "produksi",
    "konsumsi_daging_ayam"      : "konsumsi_per_kapita",
    "permintaan_daging_ayam"    : "permintaan_ekor",
    "jumlah_ternak_ayam_potong" : "jumlah_dipotong",
    "harga_baseline_ayam"       : "harga_baseline",
}

def build_statistik(df: pd.DataFrame, col_map: dict, id_komoditas: int) -> pd.DataFrame:
    subset = df[["provinsi", "tahun"] + list(col_map.keys())].copy()
    subset.rename(columns=col_map, inplace=True)
    subset["kode_wilayah"] = subset["provinsi"].map(prov_to_kode)
    subset["id_komoditas"] = id_komoditas
    return subset[["kode_wilayah", "tahun", "id_komoditas",
                   "populasi", "produksi", "konsumsi_per_kapita",
                   "permintaan_ekor", "jumlah_dipotong", "harga_baseline"]]

df_statistik = pd.concat([
    build_statistik(df_bps_full, cols_sapi, 1),
    build_statistik(df_bps_full, cols_ayam, 2),
], ignore_index=True)

print(f"✅ tr_statistik    : {len(df_statistik)} baris")
print()
print("Preview ref_wilayah:")
display(df_ref_wilayah.head())
print("\nPreview tr_statistik_peternakan:")
display(df_statistik.head(8))


---
### 1.4 Injeksi ke PostgreSQL

Urutan load:
1. Buat schema tabel di `bps_db` (jika belum ada)
2. Push 4 tabel ke `bps_db` *(database sumber BPS)*
3. Push 4 tabel yang sama ke `staging_db` *(area staging gabungan)*  
   → di staging, tabel BPS diberi prefix `bps_` agar tidak bentrok dengan tabel sumber lain


In [ ]:
# ============================================================
# CELL 13 · CREATE SCHEMA & LOAD KE POSTGRESQL
# ============================================================

DDL_BPS_DB = """
CREATE TABLE IF NOT EXISTS ref_wilayah (
    kode_wilayah  INT PRIMARY KEY,
    nama_provinsi VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS ref_komoditas (
    id_komoditas  INT PRIMARY KEY,
    nama_komoditas VARCHAR(50) NOT NULL,
    satuan_berat   VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS tr_demografi_tahunan (
    kode_wilayah    INT  NOT NULL,
    tahun           INT  NOT NULL,
    jumlah_penduduk BIGINT,
    growth_populasi DECIMAL(7,4),
    PRIMARY KEY (kode_wilayah, tahun),
    FOREIGN KEY (kode_wilayah) REFERENCES ref_wilayah(kode_wilayah)
);

CREATE TABLE IF NOT EXISTS tr_statistik_peternakan (
    kode_wilayah     INT  NOT NULL,
    tahun            INT  NOT NULL,
    id_komoditas     INT  NOT NULL,
    populasi         BIGINT,
    produksi         DECIMAL(15,2),
    konsumsi_per_kapita DECIMAL(10,2),
    permintaan_ekor  BIGINT,
    jumlah_dipotong  BIGINT,
    harga_baseline   DECIMAL(15,2),
    PRIMARY KEY (kode_wilayah, tahun, id_komoditas),
    FOREIGN KEY (kode_wilayah) REFERENCES ref_wilayah(kode_wilayah),
    FOREIGN KEY (id_komoditas) REFERENCES ref_komoditas(id_komoditas)
);
"""

# ------- Buat schema di bps_db -------
print("📦 Membuat schema di bps_db...")
try:
    with engine_bps.connect() as conn:
        for stmt in DDL_BPS_DB.strip().split(";"):
            stmt = stmt.strip()
            if stmt:
                conn.execute(text(stmt))
        conn.commit()
    print("   ✅ Schema berhasil dibuat / sudah ada.")
except Exception as e:
    print(f"   ❌ Gagal membuat schema: {e}")

# ------- Load ke bps_db -------
print()
print("📤 Memuat data ke bps_db...")
try:
    df_ref_wilayah.to_sql("ref_wilayah",            engine_bps, if_exists="append", index=False)
    df_ref_komoditas.to_sql("ref_komoditas",         engine_bps, if_exists="append", index=False)
    df_demografi.to_sql("tr_demografi_tahunan",      engine_bps, if_exists="append", index=False)
    df_statistik.to_sql("tr_statistik_peternakan",   engine_bps, if_exists="append", index=False)
    print("   ✅ 4 tabel berhasil dimuat ke bps_db")
except Exception as e:
    print(f"   ❌ Gagal load ke bps_db: {e}")

# ------- Load ke staging_db (prefix bps_) -------
print()
print("📤 Memuat data ke staging_db (prefix: bps_)...")
try:
    df_ref_wilayah.to_sql("bps_ref_wilayah",          engine_staging, if_exists="replace", index=False)
    df_ref_komoditas.to_sql("bps_ref_komoditas",       engine_staging, if_exists="replace", index=False)
    df_demografi.to_sql("bps_tr_demografi_tahunan",    engine_staging, if_exists="replace", index=False)
    df_statistik.to_sql("bps_tr_statistik_peternakan", engine_staging, if_exists="replace", index=False)
    print("   ✅ 4 tabel berhasil dimuat ke staging_db")
except Exception as e:
    print(f"   ❌ Gagal load ke staging_db: {e}")

print()
print("=" * 55)
print("  RINGKASAN EXTRACT BPS")
print("=" * 55)
print(f"  ref_wilayah            : {len(df_ref_wilayah):>5} baris")
print(f"  ref_komoditas          : {len(df_ref_komoditas):>5} baris")
print(f"  tr_demografi_tahunan   : {len(df_demografi):>5} baris")
print(f"  tr_statistik_peternakan: {len(df_statistik):>5} baris")
print("=" * 55)


---
## 2. Extract Data iSIKHNAS (MySQL → PostgreSQL)

Data iSIKHNAS tersedia di MySQL (`isikhnas_db`) yang sudah di-load via phpMyAdmin  
menggunakan file `data_peternakan.sql`.

**Tabel yang diekstrak:**

| Tabel MySQL | Nama di staging_db | Keterangan |
|---|---|---|
| `ref_hewan` | `isikhnas_ref_hewan` | Master jenis hewan |
| `ref_wilayah` | `isikhnas_ref_wilayah` | Master wilayah iSIKHNAS |
| `tr_mutasi` | `isikhnas_tr_mutasi` | Arus perpindahan ternak antar provinsi |
| `tr_laporan_sakit` | `isikhnas_tr_laporan_sakit` | Laporan hewan bergejala penyakit |
| `tr_hasil_lab` | `isikhnas_tr_hasil_lab` | Hasil uji laboratorium |
| `tr_rph` | `isikhnas_tr_rph` | Data pemotongan di RPH (berat karkas) |

> Semua tabel diberi prefix `isikhnas_` di staging_db agar tidak bentrok dengan tabel BPS.


In [ ]:
# ============================================================
# CELL 15 · EXTRACT iSIKHNAS: MySQL → staging_db
# ============================================================

TABEL_ISIKHNAS = [
    "ref_hewan",
    "ref_wilayah",
    "tr_mutasi",
    "tr_laporan_sakit",
    "tr_hasil_lab",
    "tr_rph",
]

print("🔄 Memulai extract iSIKHNAS (MySQL → staging_db)...")
print()

isikhnas_summary = {}

for tbl in TABEL_ISIKHNAS:
    staging_tbl = f"isikhnas_{tbl}"
    try:
        # Read dari MySQL
        df_tbl = pd.read_sql(f"SELECT * FROM {tbl}", engine_mysql)
        n_rows = len(df_tbl)

        # Push ke staging_db
        df_tbl.to_sql(staging_tbl, engine_staging, if_exists="replace", index=False)

        isikhnas_summary[tbl] = n_rows
        print(f"  ✅ {tbl:<25} → {staging_tbl:<35} ({n_rows:>6} baris)")

    except Exception as e:
        isikhnas_summary[tbl] = "ERROR"
        print(f"  ❌ {tbl:<25} → GAGAL: {e}")

print()
print("=" * 55)
print("  RINGKASAN EXTRACT iSIKHNAS")
print("=" * 55)
for tbl, info in isikhnas_summary.items():
    status = f"{info} baris" if isinstance(info, int) else info
    print(f"  {tbl:<28}: {status}")
print("=" * 55)


---
## 3. Extract Data PIHPS (Excel → PostgreSQL)

Data harga pangan harian dari **PIHPS (Panel Harga Pangan Strategis)**  
tersedia dalam file `final_data.xlsx`.

**Variabel yang diekstrak:**

| Variabel | Keterangan |
|---|---|
| `Provinsi` | Nama provinsi |
| `Kota` | Nama kota/kabupaten |
| `Nama_komoditas` | Nama komoditas (mis. Daging Sapi, Daging Ayam) |
| `Jenis_pasar` | Pasar tradisional / modern / pedagang besar |
| `Harga` | Harga dalam Rp/kg |
| `Waktu` | Tanggal pencatatan harga |

> File dicari dari beberapa lokasi yang mungkin. Sesuaikan `PIHPS_FILE_PATH` di Cell 1 jika perlu.


In [ ]:
# ============================================================
# CELL 17 · EXTRACT PIHPS: Excel → staging_db
# ============================================================
import os

# Cari file dari beberapa lokasi yang mungkin
possible_paths = [
    PIHPS_FILE_PATH,
    os.path.join(os.path.dirname(os.getcwd()), "DATA", "PIHPS", "final_data.xlsx"),
    os.path.join(os.path.dirname(os.getcwd()), "DATA", "final_data.xlsx"),
    "final_data.xlsx",
    "../final_data.xlsx",
]

df_pihps = None
found_path = None

for path in possible_paths:
    if os.path.exists(path):
        found_path = path
        break

print("🔄 Memulai extract PIHPS...")
print()

if found_path:
    print(f"  📂 File ditemukan: {found_path}")
    try:
        df_pihps = pd.read_excel(found_path)
        print(f"  ✅ File berhasil dibaca: {df_pihps.shape[0]} baris × {df_pihps.shape[1]} kolom")
        print(f"     Kolom: {list(df_pihps.columns)}")
        print()
        display(df_pihps.head(5))

        # Push ke staging_db
        df_pihps.to_sql("pihps_harga_pasar", engine_staging, if_exists="replace", index=False)
        print(f"\n  ✅ Data PIHPS berhasil dimuat ke staging_db → tabel: pihps_harga_pasar")

    except Exception as e:
        print(f"  ❌ Gagal membaca/muat file PIHPS: {e}")
else:
    print("  ❌ File PIHPS tidak ditemukan di semua lokasi yang dicek:")
    for p in possible_paths:
        print(f"     • {p}")
    print()
    print("  ⚠️  Sesuaikan variabel PIHPS_FILE_PATH di Cell 1 dengan lokasi file Anda.")

print()
print("=" * 55)
print("  RINGKASAN EXTRACT PIHPS")
print("=" * 55)
if df_pihps is not None:
    print(f"  Total baris  : {len(df_pihps)}")
    print(f"  Total kolom  : {len(df_pihps.columns)}")
    print(f"  Target tabel : pihps_harga_pasar (staging_db)")
else:
    print("  Status       : GAGAL — file tidak ditemukan")
print("=" * 55)


---
## Ringkasan & Validasi Akhir Fase Extract

Verifikasi bahwa semua tabel sudah masuk ke `staging_db` dengan jumlah baris yang benar.


In [ ]:
# ============================================================
# CELL 19 · VALIDASI AKHIR — Cek staging_db
# ============================================================

print("=" * 65)
print("  VALIDASI AKHIR FASE EXTRACT")
print("  Target database: staging_db (PostgreSQL)")
print("=" * 65)

try:
    inspector = inspect(engine_staging)
    tabel_staging = sorted(inspector.get_table_names())
    print(f"  Total tabel di staging_db: {len(tabel_staging)}")
    print()
    print(f"  {'Nama Tabel':<40} {'Jumlah Baris':>12}")
    print("  " + "-" * 54)

    total_rows = 0
    for tbl in tabel_staging:
        try:
            result = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", engine_staging)
            n = int(result["n"].iloc[0])
            total_rows += n
            source_tag = ""
            if tbl.startswith("bps_"):
                source_tag = "📊"
            elif tbl.startswith("isikhnas_"):
                source_tag = "🐄"
            elif tbl.startswith("pihps_"):
                source_tag = "💰"
            print(f"  {source_tag} {tbl:<38} {n:>12,}")
        except Exception as e:
            print(f"  ❌ {tbl:<38} ERROR: {e}")

    print("  " + "-" * 54)
    print(f"  {'TOTAL KESELURUHAN':<40} {total_rows:>12,}")
    print()
    print("  Legenda: 📊 = BPS  🐄 = iSIKHNAS  💰 = PIHPS")
    print()
    print("=" * 65)
    print("  ✅ FASE EXTRACT SELESAI")
    print("     Lanjut ke: ETL_Transform_Kelompok1.ipynb")
    print("=" * 65)

except Exception as e:
    print(f"❌ Gagal membaca staging_db: {e}")
    print("   Pastikan database staging_db sudah dibuat di PostgreSQL.")
